# Causal Attention Mechanism

## Why?

Modern LLMs are based on famous game changing transformer architecture from Google proposed in the famous research paper _"[Attention is All you Need](https://arxiv.org/abs/1706.03762)"_. There were two modules `Encoder` & `Decoder`. However, modern GPT based LLMs are based on `Decoder` only tranformer architecture which predicts next token only one at a time.

```
"Your" --> "journey"
"Your journey" --> "start"
"Your journey start" --> "with"
"Your journey start with" --> "one"
"Your journey start with one" --> "step"
```

Therefore we would want the self-attention mechanism to consider only the tokens that appear prior to the current position when predicting the next token in a sequence. 

**Decoder Only Transformer Architecture (GPT i.e. Autoregressive Model)**

<img src="images/transformer-architecture-decoder-only.png" width=50% />

<img src="images/transformer-architecture-decoder-2.png" width=50% />

## What?

Causal attention, also known as masked attention, is a specialized form of self-attention. It restricts a model to only consider previous and current inputs in a sequence when processing any given token when computing attention scores. This is in contrast to the standard self-attention mechanism, which allows access to the entire input sequence at once. The standard self-attention mechanism is used in full transformer architecture which has encoder & decoder modules which comes handy for tasks like translation from one language to another language.


## Goal
Modify the standard self-attention mechanism to create a causal attention mechanism which still generates context vectors (informed vectors). 

<img src="images/causal-masked-attention-overview.png" width=50% />
 

## Recap

Let's start from the milestone we achieved in my last [computational essay](trainable-attention-essay.ipynb) i.e. abstraction of the self attention mechanism with trainable weights to capture nuanced and contextual usage of words in a natural language.

## Plan

- Prepare the simple mask.
- Apply the simple mask directly on top of attention weights.
- Normalize the values after masking to keep the row sum equals to 1.
- Simplify the process by applying mask to raw attention scores instead of attention weights and then normalize by softmax.

In [1]:
import torch
import torch.nn as nn

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T

        d_k = keys.shape[-1]
        attn_weights = torch.softmax(
            attn_scores / d_k**0.5, dim=-1
        )
        
        context_vectors = attn_weights @ values
        
        return context_vectors

## Extract Code Snippets from the SelfAttention_v2 Abstraction

This is to get the attention_weights on which we will apply the causal mask.

In [2]:
torch.manual_seed(789)

inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your     (x^0)
    [0.55, 0.87, 0.66], # journey  (x^1)
    [0.57, 0.85, 0.64], # starts   (x^2)
    [0.22, 0.58, 0.33], # with     (x^3)
    [0.77, 0.25, 0.10], # one      (x^4)
    [0.05, 0.80, 0.55]  # step     (x^5)
])

d_in = inputs.shape[1] # 3
d_out = 2

sa_v2 = SelfAttention_v2(d_in, d_out)

queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 

attn_scores = queries @ keys.T

d_k = keys.shape[-1]
attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)

print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


## Prepare the Causal Mask

In [3]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


## Apply the Causal Mask

In [4]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


## Normalize After Masking

In [5]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


## Simplify 

Instead of four step process, we can acheive the same results in three steps by working with raw attention scores instead of attention weights.

**Before Simplification**

<img src="images/causal-masked-attention-4-steps.png" width=50% />

**After Simplification**

<img src="images/causal-masked-attention-3-steps.png" width=50% />

### Trick
The softmax function converts its inputs into a probability distribution. When negative infinity values (-∞) are present in a row, the softmax function treats them as zero probability. (Mathematically, this is because e  –∞ approaches 0.)

We can implement this more efficient masking “trick” by creating a mask with 1s above the diagonal and then replacing these 1s with negative infinity (-inf) values:

In [6]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [7]:
d_k = keys.shape[-1]
attn_weights = torch.softmax(masked / d_k**0.5, dim=1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


**Note:** _This is same result as we calculated before simplification._

## Dropout in Attention Mechanism

It is a technique to prevent learning curve overfitting like overly learning certain positions for some tokens.

It is a standard technique to avoid overfitting for about 10+ years in deep learning community.

In original GPT-2 architecture implementation, drop out technique was used.

However, in modern latest LLMs, it is not being used anymore.

Dropout layer randomly mask or drop the values from the attention weight matrix and to compensate for the loss it also scale the remaining values.

<img src="images/causal-masked-attention-dropout-mask-step.png" width=50% />

In [8]:
torch.manual_seed(123)
dropout_rate = 0.5

dropout_layer = torch.nn.Dropout(dropout_rate)

In [9]:
dropout_layer(attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)

## Abstracting: Causal Attention Mechanism Implementation

In [10]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_window, qkv_bias=False, dropout_rate=0.0):
        super().__init__()
        self.d_out = d_out
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        self.dropout = nn.Dropout(dropout_rate)
        
        self.register_buffer(
           'mask',
           torch.triu(torch.ones(context_window, context_length),
           diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)   
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) 

        d_k = keys.shape[-1]
        attn_weights = torch.softmax(
            attn_scores / d_k**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        
        return context_vec

## Example: Using the Causal Attention Abstraction

Implementation should handle batches consisting of more than one input so that the CausalAttention class supports the batch outputs produced by the data loader module as internet dataset will be too large to fit into one matrix therefore input embeddings will span to multiple batches.

For simplicity, to simulate such batch inputs, we duplicate the input text example of "Your journey starts with one step":

In [11]:
import torch

inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your     (x^0)
    [0.55, 0.87, 0.66], # journey  (x^1)
    [0.57, 0.85, 0.64], # starts   (x^2)
    [0.22, 0.58, 0.33], # with     (x^3)
    [0.77, 0.25, 0.10], # one      (x^4)
    [0.05, 0.80, 0.55]  # step     (x^5)
])

batch = torch.stack((inputs, inputs), dim=0)
print(f"Input shape: {batch.shape}")

torch.manual_seed(123)
d_in = batch.shape[2]
d_out = 2
context_window = batch.shape[1]

causal_attention_mechanism = CausalAttention(d_in, d_out, context_window, dropout_rate=0.0)
context_vectors = causal_attention_mechanism(batch)

print("\nAttention Mechanism Output i.e. Context Vectors:\n", context_vectors)

Input shape: torch.Size([2, 6, 3])

Attention Mechanism Output i.e. Context Vectors:
 tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)


**Note:** _Both batches have same values as inputs were just stacked for simplicity and simulating the input data from data_loader module_

## What did we Learn?

- First we understood why causal attention mechanism is needed in LLMs.
- Then we understood what is causal attention mechanism.
- Then we understood how causal attention mechanism is achieved with code examples.
- Finally we abstarcted the causal attention mechanism to use it in other modules.